# NHTSA complaints: overview

One row per complaint (`ODINO`) from `load_complaints()`; field definitions are in
`data/raw/CMPL.txt`. Caveats from NHTSA's notes in that file:

- Since a May 2021 system change, previously blank Y/N fields read **N** and blank
  numbers read **0**, so N/0 on older complaints can mean "not recorded".
- Complaint text (`CDESCR`) is capped at 2,048 characters.
- The current year is partial.

In [1]:
import altair as alt
import polars as pl
from IPython.display import display

from nhtsa.load import load_complaints, load_rows
from nhtsa.notebook import show_texts, stats

In [2]:
rows = load_rows()
complaints = load_complaints().with_columns(pl.col("LDATE").dt.year().alias("YEAR"))

multi = complaints.select(
    (pl.col("COMPONENTS").list.len() > 1).mean().alias("components"),
    (pl.col("PRODUCTS").list.len() > 1).mean().alias("products"),
).row(0, named=True)
stats(
    {
        "complaints": f"{complaints.height:,}",
        "flat-file rows": f"{rows.height:,}",
        "received": f"{complaints['LDATE'].min()} → {complaints['LDATE'].max()}",
        "with >1 component": f"{multi['components']:.1%}",
        "with >1 vehicle/product": f"{multi['products']:.1%}",
    }
)

## Volume and source

In [3]:
# Codes from CMPL.txt field 21; the long tail is lumped into "other".
SOURCES = {
    "IVOQ": "NHTSA web site",
    "EVOQ": "hotline",
    "VOQ": "owner questionnaire",
    "MIVQ": "mobile app",
    "MAVQ": "mobile app",
    "LETR": "consumer letter",
}
by_year = (
    complaints.with_columns(
        pl.col("CMPL_TYPE").replace_strict(SOURCES, default="other").alias("source")
    )
    .group_by("YEAR", "source")
    .len()
)
alt.Chart(by_year).mark_bar().encode(
    x=alt.X("YEAR:O", title=None),
    y=alt.Y("len:Q", title="complaints"),
    color=alt.Color("source:N", sort="-y"),
    tooltip=["YEAR", "source", alt.Tooltip("len:Q", format=",")],
).properties(title="Complaints received per year, by source", width=720)

alt.Chart(...)

In [4]:
# Codes from CMPL.txt field 46.
PRODUCT_TYPES = {"V": "vehicle", "T": "tires", "E": "equipment", "C": "child restraint"}
by_type = (
    complaints["PROD_TYPE"]
    .replace_strict(PRODUCT_TYPES, default="unknown")
    .value_counts()
)
alt.Chart(by_type).mark_bar().encode(
    x=alt.X("count:Q", title="complaints"),
    y=alt.Y("PROD_TYPE:N", sort="-x", title=None),
    tooltip=[alt.Tooltip("count:Q", format=",")],
).properties(title="Product type", width=500)

alt.Chart(...)

## Components

`COMPDESC` is hierarchical (`GROUP:SUBSYSTEM:PART`). The top-level group is a
candidate label set for evaluating clusters later, but it needs cleanup first:

- Detail shrinks over time: ~85% of 1995–99 components have subsystems, ~10%
  after 2010.
- Overlapping groups coexist, e.g. `ENGINE` vs `ENGINE AND ENGINE COOLING`,
  `SERVICE BRAKES` vs `SERVICE BRAKES, HYDRAULIC`.

In [5]:
components = (
    complaints.select("ODINO", "COMPONENTS")
    .explode("COMPONENTS")
    .with_columns(pl.col("COMPONENTS").str.split(":").list.first().alias("GROUP"))
)
print(
    f"{components['GROUP'].n_unique()} top-level groups, "
    f"{components['COMPONENTS'].n_unique():,} distinct full components."
)

# A complaint with two parts in the same group counts once.
groups = components.unique(["ODINO", "GROUP"]).group_by("GROUP").len()
alt.Chart(groups.sort("len", descending=True).head(25)).mark_bar().encode(
    x=alt.X("len:Q", title="complaints"),
    y=alt.Y("GROUP:N", sort="-x", title=None),
    tooltip=["GROUP", alt.Tooltip("len:Q", format=",")],
).properties(title="Top 25 component groups", width=560)

54 top-level groups, 772 distinct full components.


alt.Chart(...)

In [6]:
per_complaint = complaints.select(
    pl.col("COMPONENTS").list.len().clip(upper_bound=6).alias("n")
)["n"].value_counts()
alt.Chart(per_complaint).mark_bar().encode(
    x=alt.X("n:O", title="components per complaint (6 = 6+)"),
    y=alt.Y("count:Q", title="complaints"),
    tooltip=[alt.Tooltip("count:Q", format=",")],
).properties(width=360)

alt.Chart(...)

## Vehicles

In [7]:
products = complaints.select("ODINO", "PRODUCTS").explode("PRODUCTS").unnest("PRODUCTS")
makes = (
    products.unique(["ODINO", "MAKETXT"])["MAKETXT"].value_counts(sort=True).head(20)
)
model_years = (
    products.select(pl.col("YEARTXT").cast(pl.Int32, strict=False))
    .filter(pl.col("YEARTXT").is_between(1975, 2027))["YEARTXT"]
    .value_counts()
)
(
    alt.Chart(makes)
    .mark_bar()
    .encode(
        x=alt.X("count:Q", title="complaints"),
        y=alt.Y("MAKETXT:N", sort="-x", title=None),
        tooltip=[alt.Tooltip("count:Q", format=",")],
    )
    .properties(title="Top 20 makes", width=320)
    | alt.Chart(model_years)
    .mark_bar()
    .encode(
        x=alt.X("YEARTXT:O", title="model year", axis=alt.Axis(labelAngle=-90)),
        y=alt.Y("count:Q", title="complaints"),
        tooltip=["YEARTXT", alt.Tooltip("count:Q", format=",")],
    )
    .properties(title="Model year", width=420)
)

alt.HConcatChart(...)

## Severity

Share of complaints reporting each outcome, by year received. Read pre-2021
values with the blank → N/0 caveat in mind. `MEDICAL_ATTN` was added in April
2014, so it's shown from 2015 on.

In [8]:
severity = (
    complaints.group_by("YEAR")
    .agg(
        crash=pl.col("CRASH").mean(),
        fire=pl.col("FIRE").mean(),
        injury=(pl.col("INJURED") > 0).mean(),
        medical=pl.col("MEDICAL_ATTN").filter(pl.col("YEAR") >= 2015).mean(),
        death=(pl.col("DEATHS") > 0).mean(),
    )
    .unpivot(index="YEAR", variable_name="outcome", value_name="share")
)
alt.Chart(severity).mark_line(point=True).encode(
    x=alt.X("YEAR:O", title=None),
    y=alt.Y("share:Q", axis=alt.Axis(format="%")),
    color="outcome:N",
    tooltip=["YEAR", "outcome", alt.Tooltip("share:Q", format=".2%")],
).properties(width=720)

alt.Chart(...)

## Complaint text

Things that will matter for TF-IDF and embeddings:

- Everything received through 2020 is all caps; from 2022 almost nothing is
  (the 2021 system change). Unnormalized, casing alone reveals the era.
- The hotline template ("TL* The contact owns a …") starts in 2007.
- Older texts often end in analyst initials like `*TR` or `*AK`.

In [9]:
text = complaints.select(
    "ODINO",
    "YEAR",
    words=pl.col("CDESCR").str.count_matches(r"\S+"),
    truncated=pl.col("CDESCR").str.len_chars() >= 2048,
    all_caps=pl.col("CDESCR").str.contains(r"[A-Z]")
    & ~pl.col("CDESCR").str.contains(r"[a-z]"),
    template=pl.col("CDESCR").str.contains(r"(?i)^(TL\*\s*)?THE CONTACT"),
)
stats(
    {
        "median words": f"{text['words'].median():.0f}",
        "hit 2,048-char cap": f"{text['truncated'].mean():.2%}",
        "all caps": f"{text['all_caps'].mean():.1%}",
        "hotline template": f"{text['template'].mean():.1%}",
    }
)

In [10]:
word_hist = (
    text.group_by((pl.col("words").clip(upper_bound=400) // 10 * 10).alias("start"))
    .agg(count=pl.len())
    .with_columns(end=pl.col("start") + 10)
)
alt.Chart(word_hist).mark_bar().encode(
    x=alt.X("start:Q", bin="binned", title="words (400 = 400+)"),
    x2="end:Q",
    y=alt.Y("count:Q", title="complaints"),
).properties(width=720, height=200)

alt.Chart(...)

In [11]:
style = (
    text.group_by("YEAR")
    .agg(pl.col("all_caps").mean(), pl.col("template").mean())
    .unpivot(index="YEAR", variable_name="style", value_name="share")
)
alt.Chart(style).mark_line(point=True).encode(
    x=alt.X("YEAR:O", title=None),
    y=alt.Y("share:Q", axis=alt.Axis(format="%")),
    color="style:N",
    tooltip=["YEAR", "style", alt.Tooltip("share:Q", format=".1%")],
).properties(title="Text style by year received", width=720)

alt.Chart(...)

## Read some complaints

Change `GROUP` or `SEED` and re-run the cell for a different sample.

In [12]:
GROUP = "AIR BAGS"  # any top-level group from the chart above
SEED = 0

ids = components.filter(pl.col("GROUP") == GROUP)["ODINO"].unique()
for r in (
    complaints.filter(pl.col("ODINO").is_in(ids.implode()))
    .sample(5, seed=SEED)
    .iter_rows(named=True)
):
    display(
        show_texts(
            f"{r['ODINO']} · received {r['LDATE']} · {', '.join(r['COMPONENTS'])}",
            r["CDESCR"],
        )
    )

## Missing values (flat-file rows)

In [13]:
missing = rows.select(pl.all().is_null().mean()).unpivot(
    variable_name="column", value_name="null_share"
)
alt.Chart(missing).mark_bar().encode(
    x=alt.X("null_share:Q", axis=alt.Axis(format="%"), title="null"),
    y=alt.Y("column:N", sort="-x", title=None),
    tooltip=["column", alt.Tooltip("null_share:Q", format=".1%")],
).properties(width=480)

alt.Chart(...)